# Prior file analysis used for analysis below: 
- subject_id	stay_id	name	gsn	duration	intime	outtime	chiefcomplaint	duration (hours)	grade of medicine administered

# File data working with for the next stage of analysis

hosp/admissions.csv - subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag


hosp/patients.csv - subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod


hosp/prescriptions.csv - subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,formulary_drug_cd,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route

hosp/drgcodes.csv - subject_id,hadm_id,drg_type,drg_code,description,drg_severity,drg_mortality

hosp/d_icd_diagnoses.csv - icd_code,icd_version,long_title

hosp/emar.csv - subject_id,hadm_id,emar_id,emar_seq,poe_id,pharmacy_id,enter_provider_id,charttime,medication,event_txt,scheduletime,storetime

hosp/hcpcsevents.csv - subject_id,hadm_id,chartdate,hcpcs_cd,seq_num,short_description

# Stage 1, 2 and 3 of the Demonstration Explained

### Stage 1
Data processing primarily to match similar of the initial DEMO dataset.

- As there will be many associated GSNs associated with the grades null values may be required and will be implemented into the file as required
- The current setup in the DEMO associated file was: GSN Freq, GSN and medicine, and normalisation

### Stage 2
Once the secret data set has been cleaned (i.e., data cleaning process)

- Any GSN with null values and a medical string must be placed into a new dataframe with a GSN, string value of the medicine and a grade.
- These GSNs will also be grouped top-down by their GSN and from here by dropping those of lower frequencies reduces workload in the case of manual grading

### Third & Final Stage

- API Integration and calculating values

# Import Statements

In [ ]:
import pandas as pd
import requests
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Working Document

In [ ]:
# ------------------------------ Stage 1: Data Processing & Initialization ------------------------------
# Load and normalize the hospital datasets for GSN-medicine matching

# Load all required datasets
admissions_df = pd.read_csv('hosp/admissions.csv')
patients_df = pd.read_csv('hosp/patients.csv')
prescriptions_df = pd.read_csv('hosp/prescriptions.csv')
drgcodes_df = pd.read_csv('hosp/drgcodes.csv')
icd_diagnoses_df = pd.read_csv('hosp/d_icd_diagnoses.csv')
emar_df = pd.read_csv('hosp/emar.csv')
hcpcs_df = pd.read_csv('hosp/hcpcsevents.csv')

# Merge admissions with prescriptions on subject_id and hadm_id
merged_df = admissions_df.merge(prescriptions_df, on=['subject_id', 'hadm_id'], how='inner')

# Normalize GSN and drug columns (lowercase, strip spaces)
merged_df['gsn'] = merged_df['gsn'].astype(str).str.lower().str.strip()
merged_df['drug'] = merged_df['drug'].astype(str).str.lower().str.strip()

# Initialize grade column as null for later assignment
merged_df['grade_of_medicine'] = pd.NA

# Display sample of merged data
print(f"Merged dataset shape: {merged_df.shape}")
print(merged_df[['subject_id', 'gsn', 'drug', 'starttime', 'stoptime']].head())

# Calculate duration for each prescription
merged_df['starttime'] = pd.to_datetime(merged_df['starttime'], errors='coerce')
merged_df['stoptime'] = pd.to_datetime(merged_df['stoptime'], errors='coerce')
merged_df['duration_hours'] = (merged_df['stoptime'] - merged_df['starttime']).dt.total_seconds() / 3600

# Filtering for main drug types, remove invalid GSN and drug entries, ensure valid durations
merged_df = merged_df[merged_df['drug_type'] == 'MAIN']
merged_df = merged_df[merged_df['gsn'] != 'nan']
merged_df = merged_df[merged_df['drug'] != 'nan']
merged_df = merged_df[merged_df['duration_hours'].notna() & (merged_df['duration_hours'] > 0)]

# selecting only required columns for GSN-medicine matching and analysis
merged_df = merged_df[['subject_id', 'hadm_id', 'gsn', 'drug', 'duration_hours', 'grade_of_medicine']]

# Renaming the 'drug' col to 'medicine' for consistency
merged_df.rename(columns={'drug': 'medicine'}, inplace=True)

# Displaying a cleaned dataset info
print(f"Cleaned merged dataset shape: {merged_df.shape}")
print(merged_df.head())

print(f"\n------------------------------Dataset ready for Stage 2 processing------------------------------\n\n")

In [ ]:
# ------------------------------ Stage 2: Identify GSNs with null grades and prepare for manual grading ------------------------------

# Filtering rows where grade is missing but medicine is present
null_grade_df = merged_df[merged_df['grade_of_medicine'].isna() & merged_df['medicine'].notna()].copy()

# Computing frequency of each GSN in this null-grade subset
gsn_freq = null_grade_df.groupby('gsn').size().reset_index(name='frequency')

# Merge frequency back into the null-grade dataframe and sort by frequency
null_grade_df = null_grade_df.merge(gsn_freq, on='gsn')
null_grade_df = null_grade_df.sort_values(by='frequency', ascending=False)

# Preparing a reduced dataframe for manual review (one row per GSN/medicine combo)
manual_review_df = (
    null_grade_df
    [['gsn', 'medicine', 'grade_of_medicine', 'frequency']]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Null-grade records: {len(null_grade_df)}")
print(f"Unique GSN/medicine combos for review: {len(manual_review_df)}")
print(manual_review_df.head())

In [ ]:
# ------------------------------ Stage 3: OpenFDA lookup for NDC -> open-source equivalent identifiers (e.g., RxCUI) to pair with GSN/medicine ------------------------------

# Normalize NDC values in prescriptions_df for lookup
def _normalize_ndc(val):
    if pd.isna(val):
        return None
    s = str(val).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

prescriptions_df['ndc_clean'] = prescriptions_df['ndc'].apply(_normalize_ndc)

# Build a small lookup list of unique NDCs to query (limit to avoid hitting API rate limits)
unique_ndcs = prescriptions_df['ndc_clean'].dropna().unique()
ndcs_to_lookup = unique_ndcs[:200]  # adjust as needed

def _fetch_openfda_for_ndc(ndc):
    url = f"https://api.fda.gov/drug/ndc.json?search=product_ndc:\"{ndc}\"&limit=1"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
        results = data.get('results', [])
        if not results:
            return None
        openfda = results[0].get('openfda', {})
        return {
            'ndc': ndc,
            'openfda_rxcui': openfda.get('rxcui'),
            'openfda_generic_name': openfda.get('generic_name'),
            'openfda_brand_name': openfda.get('brand_name'),
            'openfda_spl_set_id': openfda.get('spl_set_id'),
        }
    except Exception:
        return None

openfda_rows = []
for ndc in ndcs_to_lookup:
    row = _fetch_openfda_for_ndc(ndc)
    if row:
        openfda_rows.append(row)

openfda_df = pd.DataFrame(openfda_rows)

# ensuring openfda_df has the expected columns even if empty
expected_cols = ['ndc', 'openfda_rxcui', 'openfda_generic_name', 'openfda_brand_name', 'openfda_spl_set_id']
for c in expected_cols:
    if c not in openfda_df.columns:
        openfda_df[c] = pd.NA

# merging openFDA info back into merged_df via prescriptions ndc
merged_stage3_df = merged_df.merge(
    prescriptions_df[['subject_id', 'hadm_id', 'drug', 'ndc_clean']],
    left_on=['subject_id', 'hadm_id', 'medicine'],
    right_on=['subject_id', 'hadm_id', 'drug'],
    how='left'
).merge(
    openfda_df,
    left_on='ndc_clean',
    right_on='ndc',
    how='left'
)
# Drop columns if they exist
merged_stage3_df = merged_stage3_df.drop(columns=[c for c in ['drug', 'ndc'] if c in merged_stage3_df.columns])

print("Stage 3 merged shape:", merged_stage3_df.shape)
print(merged_stage3_df[['gsn', 'medicine', 'ndc_clean', 'openfda_rxcui', 'openfda_generic_name']].head())

In [ ]:
out_df = merged_stage3_df.copy()

# Ensure ndc/drug are normalized so the merge works (if you re-merge later)
prescriptions_df['drug_norm'] = prescriptions_df['drug'].astype(str).str.lower().str.strip()

out_df = out_df.merge(
    prescriptions_df[['subject_id', 'hadm_id', 'drug_norm', 'ndc_clean']],
    left_on=['subject_id', 'hadm_id', 'medicine'],
    right_on=['subject_id', 'hadm_id', 'drug_norm'],
    how='left',
    suffixes=('', '_presc')
)

# Using OpenFDA lookup results as the grading source (example: use rxcui numeric or presence as a proxy)
out_df['grade_numeric'] = pd.to_numeric(out_df['openfda_rxcui'], errors='coerce').fillna(0)

# Normalize duration_hours, capping outliers so 22h isn’t tiny due to huge max values
dur = out_df['duration_hours'].clip(lower=0)
p99 = dur.quantile(0.99)
dur = dur.clip(upper=p99)

min_dur, max_dur = dur.min(), dur.max()
out_df['duration_normalized'] = (dur - min_dur) / (max_dur - min_dur)

# Computing frequency/rank by medicine+gsn; treat missing gsn as its own bucket as required
combo_freq = (
    out_df
    .assign(gsn_fill=out_df['gsn'].fillna('<<missing>>'))
    .groupby(['medicine', 'gsn_fill'])
    .size()
    .reset_index(name='combo_freq')
)

out_df = out_df.merge(combo_freq, left_on=['medicine', 'gsn'], right_on=['medicine', 'gsn_fill'], how='left')
out_df['combo_freq'] = out_df['combo_freq'].fillna(0)

out_df['gsn_medicine_rank'] = (
    out_df.groupby('medicine')['combo_freq']
    .rank(method='dense', ascending=False)
    .fillna(0)
    .astype(int)
)

out_df['gsn_medicine_rank_value'] = out_df['gsn_medicine_rank']

out_df['level_of_care_encoded'] = (
    out_df['duration_normalized'] * 0.6
    + out_df['grade_numeric'] * 0.65
    + out_df['gsn_medicine_rank_value'] / 10
)

out_df.to_csv('merged_stage3_loccalc_out.csv', index=False)
out_df.head()

In [ ]:
# ------------------------------ Stage 3: NDC-priority grading + LOC scoring (openFDA not returning usable data currently) ------------------------------

# openFDA lookup produced almost no results (openfda_df empty)

# Ensure prescriptions_df has normalized drug text and ndc_clean 
# and joining NDC info back onto merged_df via normalized medicine/drug text

stage3_df = merged_df.merge(
    prescriptions_df[['subject_id', 'hadm_id', 'drug_norm', 'ndc_clean']],
    left_on=['subject_id', 'hadm_id', 'medicine'],
    right_on=['subject_id', 'hadm_id', 'drug_norm'],
    how='left'
)

# Normalize ndc_clean to treat "0", empty strings, nan as missing
stage3_df['ndc_clean'] = stage3_df['ndc_clean'].astype(str).str.strip()
stage3_df.loc[stage3_df['ndc_clean'].isin(['', 'nan', 'None', '0']), 'ndc_clean'] = pd.NA

# Grade based on NDC availability (primary signal)
stage3_df['grade_numeric'] = stage3_df['ndc_clean'].notna().astype(float)

# Frequency of each medicine+gsn combo (used for ranking/LOC)
stage3_df['gsn_fill'] = stage3_df['gsn'].fillna('<<missing>>')
combo_freq = (
    stage3_df.groupby(['medicine', 'gsn_fill'])
    .size()
    .reset_index(name='combo_freq')
)
stage3_df = stage3_df.merge(combo_freq, on=['medicine', 'gsn_fill'], how='left')
stage3_df['combo_freq'] = stage3_df['combo_freq'].fillna(0)

# Normalize duration_hours to [0,1] using 99th percentile clipping to reduce outlier impact
dur = stage3_df['duration_hours'].clip(lower=0)
p99 = dur.quantile(0.99)
dur = dur.clip(upper=p99)

scaler = MinMaxScaler()
stage3_df['duration_norm'] = scaler.fit_transform(dur.to_frame())

# Normalize combo frequency to [0,1]
freq = stage3_df['combo_freq'].clip(lower=0)
freq_min, freq_max = freq.min(), freq.max()
stage3_df['freq_norm'] = (freq - freq_min) / (freq_max - freq_min) if freq_max > freq_min else 0.0

# Compute a LOC score and scale to [0,100] (and also [0,2] for variation of choice)
stage3_df['loc_score_0_100'] = (
    stage3_df['duration_norm'] * 0.5 +
    stage3_df['grade_numeric'] * 0.3 +
    stage3_df['freq_norm'] * 0.2
) * 100
stage3_df['loc_score_0_100'] = stage3_df['loc_score_0_100'].clip(0, 100)

stage3_df['loc_score_0_2'] = stage3_df['loc_score_0_100'] / 50.0
stage3_df['loc_score_0_2'] = stage3_df['loc_score_0_2'].clip(0, 2)

# Keeping relevant output columns for downstream review
stage3_out = stage3_df[[
    'subject_id', 'hadm_id', 'gsn', 'medicine',
    'ndc_clean', 'grade_numeric',
    'duration_hours', 'duration_norm',
    'combo_freq', 'freq_norm',
    'loc_score_0_100', 'loc_score_0_2'
]]
# displaying the final output and average LOC score
print(stage3_out.head())
print(f"Average LOC on 0-100 scale: {np.average(stage3_out['loc_score_0_100']):.2f}")

In [ ]:
# Saving the stage 3 output (not required)
stage3_out.to_csv('stage3_loc_scores.csv', index=False)